In [ ]:
import torch
import torch.optim as optim
from dataset import PAD_IDX, SOS_IDX, EOS_IDX
from train import train_model
from decode import decode_dataset

In [ ]:
def write_predictions(predictions, tokenizer, filename: str):
    with open(filename, "w", encoding="utf-8") as f:
        for pred_tensor in predictions:
            tokens = pred_tensor.tolist()
            if EOS_IDX in tokens:
                tokens = tokens[:tokens.index(EOS_IDX)]
            tokens = [t for t in tokens if t not in (SOS_IDX, PAD_IDX)]
            if hasattr(tokenizer, "decode"):
                text = tokenizer.decode(tokens, skip_special_tokens=True)
            else:
                text = " ".join([str(t) for t in tokens])
            f.write(text.strip() + "\n")

In [ ]:
def apply_model_surgery(model, hi_checkpoint_path, new_tgt_vocab_size, device):
    checkpoint = torch.load(hi_checkpoint_path, map_location=device, weights_only=True)
    
    reinit_keywords = ["tgt_embed", "fc_out"]
    filtered = {k: v for k, v in checkpoint.items()
                if not any(kw in k for kw in reinit_keywords)}
    
    if hasattr(model, "tgt_embed"):
        old_embed = model.tgt_embed
        model.tgt_embed = torch.nn.Embedding(new_tgt_vocab_size, old_embed.embedding_dim)
    if hasattr(model, "fc_out"):
        old_fc = model.fc_out
        model.fc_out = torch.nn.Linear(old_fc.in_features, new_tgt_vocab_size)
    
    model.load_state_dict(filtered, strict=False)
    model.to(device)
    
    retained_params = set(filtered.keys())
    print(f"Model surgery complete: loaded {len(retained_params)} params, "
          f"re-initialized tgt_embed and fc_out for vocab_size={new_tgt_vocab_size}")
    return retained_params

In [ ]:
def set_freeze_state(model, retained_params, freeze: bool):
    for name, param in model.named_parameters():
        if name in retained_params:
            param.requires_grad = not freeze
        else:
            param.requires_grad = True

In [ ]:
def run_transfer_experiment(
    transferred_model, baseline_model, hi_checkpoint_path: str,
    new_tgt_vocab_size: int,
    train_loader, val_loader, test_loader, tgt_tokenizer,
    device: torch.device, total_epochs: int = 30, warmup_epochs: int = 2,
    tf_mode: str = "fixed"
):
    print("=========================================================")
    print("=             PART 1: THE TRANSFERRED MODEL             =")
    print("=========================================================")
    retained_params = apply_model_surgery(transferred_model, hi_checkpoint_path, new_tgt_vocab_size, device)

    print(f"\n[Phase A] Warmup: Freezing encoder/LSTM for {warmup_epochs} epochs...")
    set_freeze_state(transferred_model, retained_params, freeze=True)
    warmup_opt = optim.Adam(filter(lambda p: p.requires_grad, transferred_model.parameters()), lr=1e-3)
    
    warmup_hist = train_model(
        transferred_model, train_loader, val_loader, warmup_opt, device,
        num_epochs=warmup_epochs, save_path="transfer_warmup.pth",
        tf_mode=tf_mode, fixed_tf_ratio=0.6, tgt_tokenizer=tgt_tokenizer
    )
    
    print(f"\n[Phase B] Full fine-tuning for {total_epochs - warmup_epochs} epochs...")
    transferred_model.load_state_dict(torch.load("transfer_warmup.pth", map_location=device, weights_only=True))
    set_freeze_state(transferred_model, retained_params, freeze=False)
    
    # -----------------------------
    # Reduced LR by half
    # -----------------------------
    if hasattr(transferred_model, 'bert'):
        finetune_opt = optim.Adam([
            {'params': transferred_model.bert.parameters(), 'lr': 1e-5 / 2},
            {'params': [p for n, p in transferred_model.named_parameters() if not n.startswith('bert.')], 'lr': 5e-4 / 2}
        ])
    else:
        finetune_opt = optim.Adam(transferred_model.parameters(), lr=1.5e-4)
        
    finetune_hist = train_model(
        transferred_model, train_loader, val_loader, finetune_opt, device,
        num_epochs=(total_epochs - warmup_epochs), save_path="best_transfer_model.pth",
        tf_mode=tf_mode, fixed_tf_ratio=0.6, tgt_tokenizer=tgt_tokenizer
    )

    print("\n=========================================================")
    print("=             PART 2: THE BASELINE MODEL                =")
    print("=========================================================")
    print(f"Training baseline from scratch for {total_epochs} epochs...")
    
    if hasattr(baseline_model, 'bert'):
        baseline_opt = optim.Adam([
            {'params': baseline_model.bert.parameters(), 'lr': 1e-5},
            {'params': [p for n, p in baseline_model.named_parameters() if not n.startswith('bert.')], 'lr': 5e-4}
        ])
    else:
        baseline_opt = optim.Adam(baseline_model.parameters(), lr=3e-4)
    
    baseline_hist = train_model(
        baseline_model, train_loader, val_loader, baseline_opt, device,
        num_epochs=total_epochs, save_path="best_baseline_model.pth",
        tf_mode=tf_mode, fixed_tf_ratio=0.6, tgt_tokenizer=tgt_tokenizer
    )

    print("\n=========================================================")
    print("=             PART 3: DECODING FOR SACREBLEU            =")
    print("=========================================================")
    
    print("Decoding transfer model...")
    transferred_model.load_state_dict(torch.load("best_transfer_model.pth", map_location=device, weights_only=True))
    transferred_model.to(device)
    trans_preds, _ = decode_dataset(transferred_model, test_loader, beam_size=5, device=device)
    write_predictions(trans_preds, tgt_tokenizer, "transferred_predictions.txt")
    
    print("\nDecoding baseline model...")
    baseline_model.load_state_dict(torch.load("best_baseline_model.pth", map_location=device, weights_only=True))
    baseline_model.to(device)
    base_preds, _ = decode_dataset(baseline_model, test_loader, beam_size=5, device=device)
    write_predictions(base_preds, tgt_tokenizer, "baseline_predictions.txt")
    
    return warmup_hist, finetune_hist, baseline_hist